In [ ]:
%matplotlib inline
import pandas as pd
import scipy
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import sktime
import sklearn
from scipy.io import loadmat
from sktime.transformations.panel.rocket import Rocket
import timesynth as ts
import random
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

In [ ]:
n_steps = 60
# get original segment for given sample that is impacted by a kernel
def get_original_segment(sample, impacted_k_weights,
    impacted_k_bias,
    impacted_k_dilation,
    impacted_k_padding,
    impacted_k_lengths):
    sample_data = test_X[sample, 0, :]

    # check for padding and apply to the segment if needed on both left and right sides
    if impacted_k_padding != 0:
        padding_left = ((impacted_k_lengths - 1) * impacted_k_dilation) // 2
        padding_right = ((impacted_k_lengths - 1) * impacted_k_dilation) - padding_left
        padded_segment = np.pad(sample_data, (padding_left, padding_right), mode='constant')
    else:
        padded_segment = sample_data.copy()
        
    conv = np.zeros(n_steps)
    for i in range(n_steps):
        indices = i + np.arange(impacted_k_lengths) * impacted_k_dilation
        if np.all(indices < len(padded_segment)):
            conv[i] = np.sum(padded_segment[indices] * impacted_k_weights)
    return conv + impacted_k_bias

# get injected segment
def get_injected_segment(sample, impacted_k_weights,
    impacted_k_bias,
    impacted_k_dilation,
    impacted_k_padding,
    impacted_k_lengths):
    sample_data = test_X2[sample, 0, :]

    # check for padding and apply to the segment if needed on both left and right sides
    if impacted_k_padding != 0:
        padding_left = ((impacted_k_lengths - 1) * impacted_k_dilation) // 2
        padding_right = ((impacted_k_lengths - 1) * impacted_k_dilation) - padding_left
        padded_segment = np.pad(sample_data, (padding_left, padding_right), mode='constant')
    else:
        padded_segment = sample_data.copy()

    conv = np.zeros(n_steps)
    for i in range(n_steps):
        indices = i + np.arange(impacted_k_lengths) * impacted_k_dilation
        if np.all(indices < len(padded_segment)):
            conv[i] = np.sum(padded_segment[indices] * impacted_k_weights)
    return conv + impacted_k_bias


In [ ]:
# spike data generation
for i in range(100):
    interval = 60000
    n_samples = 1000
    splits = 60
    X = np.zeros((n_samples, 1, interval))
    y = np.zeros(n_samples)

    time_sampler = ts.TimeSampler(stop_time=interval)
    regular_times = time_sampler.sample_regular_time(num_points = interval)

    narma_signal = ts.signals.NARMA(order= 2, seed = i)
    timeseries = ts.TimeSeries(signal_generator=narma_signal)

    sample, signals, errors = timeseries.sample(regular_times)
    rand = random.random()

    X = sample.reshape(n_samples,1,splits)
    train_X = X[0:700]
    test_X = X[700:]

    X2 = X.copy()

    spike_mask = np.zeros((n_samples, splits))

    for sample in range(n_samples):
        num_spikes = np.random.poisson(2)
        locations = np.random.choice(splits, size=num_spikes, replace=False)

        spike_mask[sample, locations] = 1
        X2[sample,0,locations] += 0.2

    train_X2 = X2[0:700]
    test_X2 = X2[700:]

    # Assuming X and X2 both have the shape (1000, 1, 60)
    # Axis=0 stacks them along the sample/row dimension
    train_X3 = np.concatenate((train_X, train_X2), axis=0)

    print("New shape of X3:", train_X3.shape)

    test_X3 = np.concatenate((test_X, test_X2), axis=0)

    print("New shape of X3:", test_X3.shape)

    # Expected output: (2000, 1, 60)
    y = [0] * 700 + [1] * 700
    # fit the rocket transformer on sober drive data
    trf = Rocket(num_kernels=1000, random_state = 1234)
    rocket_features = trf.fit(train_X3)
    transformed = rocket_features.transform(train_X)
    transformed

    transformed2 = rocket_features.transform(train_X2)
    transformed2

    feats = rocket_features.transform(train_X3)
    feats

    feats.to_csv('C:/Users/satavans/Downloads/spike_files_0.2spike/train_files/spike_rocket' + str(i) + '.csv', index=False)
    pd.DataFrame(y).to_csv('C:/Users/satavans/Downloads/spike_files_0.2spike/train_files/spike_y' + str(i) + '.csv', index=False)
    np.save("C:/Users/satavans/Downloads/spike_files_0.2spike/masks/spike_mask_" + str(i) + ".npy", spike_mask)
    np.save("C:/Users/satavans/Downloads/spike_files_0.2spike/train_raw/X3_" + str(i) + ".npy", train_X3)
    np.save("C:/Users/satavans/Downloads/spike_files_0.2spike/test_raw/X_" + str(i) + ".npy", test_X)
    np.save("C:/Users/satavans/Downloads/spike_files_0.2spike/test_raw/X2_" + str(i) + ".npy", test_X2)
    

In [ ]:
precision_all = []
recall_all = []
fdr_all = []
f1_all = []
# FDF results 
for i in range(100):
    test_X = np.load("C:/Users/satavans/Downloads/spike_files_0.2spike/test_raw/X_" + str(i) + ".npy")
    test_X2 = np.load("C:/Users/satavans/Downloads/spike_files_0.2spike/test_raw/X2_" + str(i) + ".npy")
    train_X3 = np.load("C:/Users/satavans/Downloads/spike_files_0.2spike/train_raw/X3_" + str(i) + ".npy")
    # fit the rocket transformer on sober drive data
    trf = Rocket(num_kernels=1000, random_state = 1234)
    rocket_features = trf.fit(train_X3)
    
    kernels = pd.read_csv("C:/Users/satavans/Downloads/spike_files_0.2spike/kernels_0.0001/spike_kernels" + str(i) + ".csv")
    selected_kernels = kernels['selected_features']
    # print(len(selected_kernels))
    # getting the kernel params
    kernel_weights = trf.kernels[0]
    kernel_lengths = trf.kernels[1]
    kernel_biases = trf.kernels[2]
    kernel_dilations = trf.kernels[3]
    kernel_paddings = trf.kernels[4]

    TP = FP = FN = 0
    total_time_points = test_X.shape[2]
    for sample in range(test_X.shape[0]):
        # 1. Get the exact coordinates that changed
        exact_changed = np.where(np.abs(test_X[sample,0,:] - test_X2[sample,0,:]) > 0)[0]
        changed_set = set(exact_changed)

        # 2. LENIENT SEEDING: Expand the changed target region by +/- 1 time step
        lenient_changed_set = set()
        for idx in exact_changed:
            lenient_changed_set.add(idx)
            if idx > 0:
                lenient_changed_set.add(idx - 1)
            if idx < total_time_points - 1:
                lenient_changed_set.add(idx + 1)

        # Gather flags from any kernel
        # Gather flags from any kernel
        any_kernel_flagged = set()
        all_point_diffs = {}
        for kernel in selected_kernels:
                    kernel_start = np.sum(kernel_lengths[:kernel])
                    kernel_end = kernel_start + kernel_lengths[kernel]

                    impacted_k_weights = kernel_weights[kernel_start:kernel_end]
                    impacted_k_bias = kernel_biases[kernel]
                    impacted_k_dilation = kernel_dilations[kernel]
                    impacted_k_padding = kernel_paddings[kernel]
                    impacted_k_lengths = kernel_lengths[kernel]

                    injected = get_injected_segment(sample, impacted_k_weights,
                                                    impacted_k_bias,
                                                    impacted_k_dilation,
                                                    impacted_k_padding,
                                                    impacted_k_lengths)

                    diff = np.abs(injected)
                    # store maximum diff seen for each timepoint
                    for idx, value in enumerate(diff):
                        if idx not in all_point_diffs:
                            all_point_diffs[idx] = value
                        else:
                            all_point_diffs[idx] = all_point_diffs.get(idx, 0) + value


        # get top k (# of spikes) points
        n_points = len(all_point_diffs)
        n_keep = len(changed_set)
        # len(changed_set)
        any_kernel_flagged = set(
            sorted(all_point_diffs, key=all_point_diffs.get, reverse=True)[:n_keep]
        )
        # print(len(any_kernel_flagged))
        # True Positives: Flagged points that land inside the lenient (expanded) target zone
        lenient_tp_set = any_kernel_flagged & lenient_changed_set
        # TP += len(lenient_tp_set)

        # False Positives: Flagged points completely outside the lenient zone
        # FP += len(any_kernel_flagged - lenient_changed_set)

        # False Negatives: Actual changed points that were missed (even with the lenient zone)
        # To keep your metric behavior balanced, we look at which *exact* changes were missed completely
        matched_predictions = set()

        for true_idx in sorted(changed_set):
            matched = False
            
            for pred_idx in any_kernel_flagged:
                if pred_idx not in matched_predictions and abs(true_idx - pred_idx) <= 1:
                    TP += 1
                    matched_predictions.add(pred_idx)
                    matched = True
                    break

            if not matched:
                FN += 1

        for pred_idx in sorted(any_kernel_flagged):
            if pred_idx not in matched_predictions:
                FP += 1
        # FN += len(changed_set - any_kernel_flagged)
        #print(TP)
        #print(FN)
    # Safe metric calculations
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    FDR = FP / (TP + FP) if (TP + FP) > 0 else 0.0
    precision_all.append(precision)
    recall_all.append(recall)
    f1_all.append(f1)
    fdr_all.append(FDR)
    

    print(
        f"Lenient Logic Result: "
        f"Precision={precision:.3f}, "
        f"Recall={recall:.3f}, "
        f"F1={f1:.3f}, "
        f"FDR={FDR:.3f}"
    )



In [ ]:
df = pd.DataFrame(
    {
        'recall': recall_all,
        'false discovery rate': fdr_all,
        'precision': precision_all,
        'f1': f1_all
    }
)

df.to_csv("C:/Users/satavans/Downloads/spike_files_0.2spike/results/metrics_topk_0.0001.csv", index = False)

In [ ]:
# generating shapelet data 
# wavelet experiment
sin_wave = 0.5 * np.sin(np.linspace(0, 2* np.pi, 10))
noise = np.random.normal(0,0.05, sin_wave.shape)
sin_wave2 = sin_wave + noise
plt.plot(sin_wave)

for i in range(100):
    interval = 60000
    n_samples = 1000
    splits = 60
    X = np.zeros((n_samples, 1, interval))
    y = np.zeros(n_samples)

    time_sampler = ts.TimeSampler(stop_time=interval)
    regular_times = time_sampler.sample_regular_time(num_points = interval)

    narma_signal = ts.signals.NARMA(order= 2, seed = i)
    timeseries = ts.TimeSeries(signal_generator=narma_signal)

    sample, signals, errors = timeseries.sample(regular_times)
    rand = random.random()

    X = sample.reshape(n_samples,1,splits)
    train_X = X[0:700]
    test_X = X[700:]

    X2 = X.copy()

    spike_mask = np.zeros((n_samples, splits))

    for sample in range(n_samples):
        location = np.random.choice(splits - 10, size = 1, replace = False)
        for j in range(10):
            X2[sample][0][location + j] = X[sample][0][location + j] + sin_wave2[j]

    train_X2 = X2[0:700]
    test_X2 = X2[700:]

    # Assuming X and X2 both have the shape (1000, 1, 60)
    # Axis=0 stacks them along the sample/row dimension
    train_X3 = np.concatenate((train_X, train_X2), axis=0)

    print("New shape of X3:", train_X3.shape)

    test_X3 = np.concatenate((test_X, test_X2), axis=0)

    print("New shape of X3:", test_X3.shape)

    # Expected output: (2000, 1, 60)
    y = [0] * 700 + [1] * 700
    # fit the rocket transformer on sober drive data
    trf = Rocket(num_kernels=1000, random_state = 1234)
    rocket_features = trf.fit(train_X3)
    transformed = rocket_features.transform(train_X)
    transformed

    transformed2 = rocket_features.transform(train_X2)
    transformed2

    feats = rocket_features.transform(train_X3)
    feats

    feats.to_csv('C:/Users/satavans/Downloads/shapelet_files/train_files/shapelet_rocket' + str(i) + '.csv', index=False)
    pd.DataFrame(y).to_csv('C:/Users/satavans/Downloads/shapelet_files/train_files/shapelet_y' + str(i) + '.csv', index=False)
    np.save("C:/Users/satavans/Downloads/shapelet_files/masks/shapelet_mask_" + str(i) + ".npy", spike_mask)
    np.save("C:/Users/satavans/Downloads/shapelet_files/train_raw/X3_" + str(i) + ".npy", train_X3)
    np.save("C:/Users/satavans/Downloads/shapelet_files/test_raw/X_" + str(i) + ".npy", test_X)
    np.save("C:/Users/satavans/Downloads/shapelet_files/test_raw/X2_" + str(i) + ".npy", test_X2)
    

# SHAPELET

In [ ]:
tpr_all = []
overlap_all_mean = []
overlap_all_90p = []
overlap_all_50p = []
overlap_all_10p = []
overlap_all_0p = []
precision_all = []
recall_all = []
f1_all = []
fdr_all = []
overlap = []
# fdf shapelet results
for i in range(100):
    test_X = np.load("C:/Users/satavans/Downloads/shapelet_files/test_raw/X_" + str(i) + ".npy")
    test_X2 = np.load("C:/Users/satavans/Downloads/shapelet_files/test_raw/X2_" + str(i) + ".npy")
    train_X3 = np.load("C:/Users/satavans/Downloads/shapelet_files/train_raw/X3_" + str(i) + ".npy")
    # fit the rocket transformer on sober drive data
    trf = Rocket(num_kernels=1000, random_state = 1234)
    rocket_features = trf.fit(train_X3)
    
    kernels = pd.read_csv("C:/Users/satavans/Downloads/shapelet_files/kernels/shapelet_kernels" + str(i) + ".csv")
    selected_kernels = kernels['selected_features']

    # getting the kernel params
    kernel_weights = trf.kernels[0]
    kernel_lengths = trf.kernels[1]
    kernel_biases = trf.kernels[2]
    kernel_dilations = trf.kernels[3]
    kernel_paddings = trf.kernels[4]

    TP = FP = FN = TN = 0
    for sample in range(test_X.shape[0]):
        # 1. Get the exact coordinates that changed
        exact_changed = np.where(np.abs(test_X[sample,0,:] - test_X2[sample,0,:]) > 0)[0]
        changed_set = set(exact_changed)

        # Gather flags from any kernel
        any_kernel_flagged = set()
        all_point_diffs = {}
        for kernel in selected_kernels:
            kernel_start = np.sum(kernel_lengths[:kernel])
            kernel_end = kernel_start + kernel_lengths[kernel]

            impacted_k_weights = kernel_weights[kernel_start:kernel_end]
            impacted_k_bias = kernel_biases[kernel]
            impacted_k_dilation = kernel_dilations[kernel]
            impacted_k_padding = kernel_paddings[kernel]
            impacted_k_lengths = kernel_lengths[kernel]

            injected = get_injected_segment(sample, impacted_k_weights,
                                            impacted_k_bias,
                                            impacted_k_dilation,
                                            impacted_k_padding,
                                            impacted_k_lengths)

            diff = np.abs(injected)
            # store maximum diff seen for each timepoint
            for idx, value in enumerate(diff):
                if idx not in all_point_diffs:
                    all_point_diffs[idx] = value
                else:
                    all_point_diffs[idx] = all_point_diffs.get(idx, 0) + value


        # get top 10 timepoints
        any_kernel_flagged = set(
            sorted(all_point_diffs, key=all_point_diffs.get, reverse=True)[:10]
        )

        tp_current = len(any_kernel_flagged & changed_set)
        TP += len(any_kernel_flagged & changed_set)
        FP += len(any_kernel_flagged - changed_set)
        FN += len(changed_set - any_kernel_flagged)
        overlap.append(tp_current/10)

        # union += len(changed | any_kernel_flagged)
        # FP += len(any_kernel_flagged - changed)
        # FN += len(changed - any_kernel_flagged)
        # print(changed & any_kernel_flagged)


    # Safe metric calculations
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    FDR = FP / (TP + FP) if (TP + FP) > 0 else 0.0
    precision_all.append(precision)
    recall_all.append(recall)
    f1_all.append(f1)
    fdr_all.append(FDR)
    tpr_all.append(TP / (TP + FN))
    overlap_all_mean.append(np.mean(overlap))
    overlap_all_90p.append(np.percentile(overlap, 90))
    overlap_all_50p.append(np.percentile(overlap, 50))
    overlap_all_10p.append(np.percentile(overlap, 10))
    overlap_all_0p.append(np.percentile(overlap, 0))

    

    print(
        f"Lenient Logic Result: "
        f"Precision={precision:.3f}, "
        f"Recall={recall:.3f}, "
        f"F1={f1:.3f}, "
        f"FDR={FDR:.3f}"
    )



In [ ]:
np.mean(f1_all)
np.percentile(f1_all, [90, 50, 10, 0])

In [ ]:
tpr_all = []
overlap_all_mean = []
overlap_all_90p = []
overlap_all_50p = []
overlap_all_10p = []
overlap_all_0p = []
precision_all = []
recall_all = []
f1_all = []
fdr_all = []
overlap = []
for i in range(100):
    test_X = np.load("C:/Users/satavans/Downloads/shapelet_files/test_raw/X_" + str(i) + ".npy")
    test_X2 = np.load("C:/Users/satavans/Downloads/shapelet_files/test_raw/X2_" + str(i) + ".npy")
    train_X3 = np.load("C:/Users/satavans/Downloads/shapelet_files/train_raw/X3_" + str(i) + ".npy")
    # fit the rocket transformer on sober drive data
    trf = Rocket(num_kernels=1000, random_state = 1234)
    rocket_features = trf.fit(train_X3)
    
    kernels = pd.read_csv("C:/Users/satavans/Downloads/shapelet_files/kernels_0.01/shapelet_kernels" + str(i) + ".csv")
    selected_kernels = kernels['selected_features']

    # getting the kernel params
    kernel_weights = trf.kernels[0]
    kernel_lengths = trf.kernels[1]
    kernel_biases = trf.kernels[2]
    kernel_dilations = trf.kernels[3]
    kernel_paddings = trf.kernels[4]

    TP = FP = FN = TN = 0
    for sample in range(test_X.shape[0]):
        # 1. Get the exact coordinates that changed
        exact_changed = np.where(np.abs(test_X[sample,0,:] - test_X2[sample,0,:]) > 0)[0]
        changed_set = set(exact_changed)

        # Gather flags from any kernel
        any_kernel_flagged = set()
        all_point_diffs = {}
        for kernel in selected_kernels:
            kernel_start = np.sum(kernel_lengths[:kernel])
            kernel_end = kernel_start + kernel_lengths[kernel]

            impacted_k_weights = kernel_weights[kernel_start:kernel_end]
            impacted_k_bias = kernel_biases[kernel]
            impacted_k_dilation = kernel_dilations[kernel]
            impacted_k_padding = kernel_paddings[kernel]
            impacted_k_lengths = kernel_lengths[kernel]

            baseline = get_original_segment(sample, impacted_k_weights,
                                            impacted_k_bias,
                                            impacted_k_dilation,
                                            impacted_k_padding,
                                            impacted_k_lengths)
            injected = get_injected_segment(sample, impacted_k_weights,
                                            impacted_k_bias,
                                            impacted_k_dilation,
                                            impacted_k_padding,
                                            impacted_k_lengths)

            diff = np.abs(injected - baseline)
            # store maximum diff seen for each timepoint
            for idx, value in enumerate(diff):
                if idx not in all_point_diffs:
                    all_point_diffs[idx] = value
                else:
                    all_point_diffs[idx] = all_point_diffs.get(idx, 0) + value


        # get top 10 timepoints
        any_kernel_flagged = set(
            sorted(all_point_diffs, key=all_point_diffs.get, reverse=True)[:10]
        )

        tp_current = len(any_kernel_flagged & changed_set)
        TP += len(any_kernel_flagged & changed_set)
        FP += len(any_kernel_flagged - changed_set)
        FN += len(changed_set - any_kernel_flagged)
        overlap.append(tp_current/10)

        # union += len(changed | any_kernel_flagged)
        # FP += len(any_kernel_flagged - changed)
        # FN += len(changed - any_kernel_flagged)
        # print(changed & any_kernel_flagged)


    # Safe metric calculations
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    FDR = FP / (TP + FP) if (TP + FP) > 0 else 0.0
    precision_all.append(precision)
    recall_all.append(recall)
    f1_all.append(f1)
    fdr_all.append(FDR)
    tpr_all.append(TP / (TP + FN))
    overlap_all_mean.append(np.mean(overlap))
    overlap_all_90p.append(np.percentile(overlap, 90))
    overlap_all_50p.append(np.percentile(overlap, 50))
    overlap_all_10p.append(np.percentile(overlap, 10))
    overlap_all_0p.append(np.percentile(overlap, 0))

    

    print(
        f"Lenient Logic Result: "
        f"Precision={precision:.3f}, "
        f"Overlap = {np.mean(overlap_all_mean)}, "
        f"Recall={recall:.3f}, "
        f"F1={f1:.3f}, "
        f"FDR={FDR:.3f}"
    )



In [ ]:
df = pd.DataFrame(
    {
        'recall': recall_all,
        'false discovery rate': fdr_all,
        'precision': precision_all,
        'overlap_mean': overlap_all_mean,
        'overlap_90p': overlap_all_90p,
        'overlap_50p': overlap_all_90p,
        'overlap_10p': overlap_all_90p,
        'overlap_0p': overlap_all_90p,
        'f1': f1_all
    }
)

df.to_csv("C:/Users/satavans/Downloads/shapelet_files/results/metrics_top10_0.01.csv", index = False)

In [ ]:
import shap
# 1. Initialize ROCKET and the classifier
rocket = Rocket(num_kernels=1000) 
classifier = LogisticRegression(max_iter= 2000)

# 2. Fit pipeline (Assuming X_train is your time series data)
X_train_transformed = rocket.fit_transform(train_X3)
classifier.fit(X_train_transformed, y)

# 2. Define the wrapper function 
def rocket_predict(X_2d):
    # SHAP passes (samples, time_steps). Inject the middle '1' dimension back for 3D ROCKET
    X_3d = X_2d.reshape(-1, 1, 60)
    X_tf = rocket.transform(X_3d)
    return classifier.decision_function(X_tf) 

# 4. Initialize KernelExplainer with a background dataset (often a KMeans summary of train data)
# Note: For time series, explaining the raw points requires careful consideration of feature independence
X_train_2d = train_X3.squeeze(axis=1) 
background_data = shap.sample(X_train_2d[0:1400], 100)
explainer = shap.KernelExplainer(rocket_predict, background_data)

# 5. Calculate SHAP values for a specific instance
X_test_2d = test_X2.squeeze(axis=1)
shap_values = explainer.shap_values(X_test_2d[0:1])

# KERNELSHAP

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import shap
# kernelSHAP results spike
precision_all = []
recall_all = []
fdr_all = []
f1_all = []
for i in range(100):
    test_X = np.load("C:/Users/satavans/Downloads/spike_files/test_raw/X_" + str(i) + ".npy")
    test_X2 = np.load("C:/Users/satavans/Downloads/spike_files/test_raw/X2_" + str(i) + ".npy")
    train_X3 = np.load("C:/Users/satavans/Downloads/spike_files/train_raw/X3_" + str(i) + ".npy")
    y = pd.read_csv('C:/Users/satavans/Downloads/spike_files/train_files/spike_y' + str(i) + '.csv')['0']
    # fit the rocket transformer on sober drive data
    trf = Rocket(num_kernels=1000, random_state = 1234)
    classifier = LogisticRegression(max_iter = 2000)
    # 2. Fit pipeline (Assuming X_train is your time series data)
    X_train_transformed = trf.fit_transform(train_X3)
    scaler = StandardScaler()
    X_train_transformed = scaler.fit_transform(X_train_transformed)
    classifier.fit(X_train_transformed, y)

    # 2. Define the wrapper function 
    def rocket_predict(X_2d):
        # SHAP passes (samples, time_steps). Inject the middle '1' dimension back for 3D ROCKET
        X_3d = X_2d.reshape(-1, 1, 60)
        X_tf = trf.transform(X_3d) 
        X_tf = scaler.transform(X_tf)
        return classifier.decision_function(X_tf) 

    # 4. Initialize KernelExplainer with a background dataset (often a KMeans summary of train data)
    # Note: For time series, explaining the raw points requires careful consideration of feature independence
    X_train_2d = train_X3.squeeze(axis=1) 
    background_data = shap.sample(X_train_2d[0:1400], 50)
    explainer = shap.KernelExplainer(rocket_predict, background_data)

    # 5. Calculate SHAP values for a specific instance
    X_test_2d = test_X2.squeeze(axis=1)

    TP = FP = FN = 0
    total_time_points = test_X.shape[2]
    for sample in range(test_X.shape[0]):
        # 1. Get the exact coordinates that changed
        exact_changed = np.where(np.abs(test_X[sample,0,:] - test_X2[sample,0,:]) > 0)[0]
        changed_set = set(exact_changed)

        # 2. LENIENT SEEDING: Expand the changed target region by +/- 1 time step
        lenient_changed_set = set()
        for idx in exact_changed:
            lenient_changed_set.add(idx)
            if idx > 0:
                lenient_changed_set.add(idx - 1)
            if idx < total_time_points - 1:
                lenient_changed_set.add(idx + 1)

        # 2. Extract continuous SHAP values for this sample and take absolute magnitudes
        sample_shap = np.abs(explainer.shap_values(X_test_2d[sample]))

        # 3. Apply the matching 95th percentile cutoff threshold
        shap_flagged = set(np.where(sample_shap > (np.sort(sample_shap)[::-1])[len(changed_set)])[0])
        # set(np.where(sample_shap > 0)[0])
        # 4. Calculate confusion metrics
        s_tp = len(lenient_changed_set & shap_flagged)
        s_fp = len(shap_flagged - lenient_changed_set)
        s_fn = len(changed_set - shap_flagged)
            
        TP += s_tp

        # False Positives: Flagged points completely outside the lenient zone
        FP += s_fp

        # False Negatives: Actual changed points that were missed (even with the lenient zone)
        # To keep your metric behavior balanced, we look at which *exact* changes were missed completely
        FN += s_fn
        print(s_tp)
        print(s_fn)
    # Safe metric calculations
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    FDR = FP / (TP + FP) if (TP + FP) > 0 else 0.0
    precision_all.append(precision)
    recall_all.append(recall)
    f1_all.append(f1)
    fdr_all.append(FDR)

    

    print(
        f"Lenient Logic Result: "
        f"Precision={precision:.3f}, "
        f"Recall={recall:.3f}, "
        f"F1={f1:.3f}, "
        f"FDR={FDR:.3f}"
    )



In [ ]:
df = pd.DataFrame(
    {
        'recall': recall_all,
        'false discovery rate': fdr_all,
        'precision': precision_all,
        'f1': f1_all
    }
)
df.to_csv("C:/Users/satavans/Downloads/spike_files/shap_results/metrics_greaterthan0.csv", index = False)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from lime.lime_tabular import LimeTabularExplainer

precision_all = []
recall_all = []
fdr_all = []
f1_all = []
# LIME results spike
for i in range(100):
    test_X = np.load("C:/Users/satavans/Downloads/spike_files/test_raw/X_" + str(i) + ".npy")
    test_X2 = np.load("C:/Users/satavans/Downloads/spike_files/test_raw/X2_" + str(i) + ".npy")
    train_X3 = np.load("C:/Users/satavans/Downloads/spike_files/train_raw/X3_" + str(i) + ".npy")
    y = pd.read_csv('C:/Users/satavans/Downloads/spike_files/train_files/spike_y' + str(i) + '.csv')['0']
    # fit the rocket transformer on sober drive data
    trf = Rocket(num_kernels=1000, random_state = 1234)
    classifier = LogisticRegression(max_iter = 2000)
    # 2. Fit pipeline (Assuming X_train is your time series data)
    X_train_transformed = trf.fit_transform(train_X3)
    scaler = StandardScaler()
    X_train_transformed = scaler.fit_transform(X_train_transformed)
    classifier.fit(X_train_transformed, y)

   # 3. Define the wrapper function 
    def rocket_predict(X_2d):
        # LIME passes (samples, time_steps). Inject the middle '1' dimension back for 3D ROCKET
        X_3d = X_2d.reshape(-1, 1, 60)
        X_tf = trf.transform(X_3d)
        X_tf = scaler.transform(X_tf)
        # LIME expects a 2D array of probabilities [prob_class_0, prob_class_1]
        # Use predict_proba instead of decision_function for classification
        return classifier.predict_proba(X_tf) 

    # 4. Prepare 2D background data and initialize LimeTabularExplainer
    # Squeeze 3D train data to 2D (samples, time_steps)
    X_train_2d = train_X3.squeeze(axis=1) 

    # Feature names represent each timestamp (from 0 to 59)
    feature_names = [f"time_{i}" for i in range(60)]

    explainer = LimeTabularExplainer(
        training_data=X_train_2d,       # Used to calculate mean/variance for feature perturbation
        mode="classification",          # Explaining a classifier
        feature_names=feature_names,                
        feature_selection = 'none'
    )

    # 5. Calculate LIME values for a specific instance
    # Squeeze 3D test data to 2D (samples, time_steps)
    X_test_2d = test_X2.squeeze(axis=1)

    

    TP = FP = FN = 0
    total_time_points = test_X.shape[2]
    for sample in range(test_X.shape[0]):
        # 1. Get the exact coordinates that changed
        exact_changed = np.where(np.abs(test_X[sample,0,:] - test_X2[sample,0,:]) > 0)[0]
        changed_set = set(exact_changed)

        # 2. LENIENT SEEDING: Expand the changed target region by +/- 1 time step
        lenient_changed_set = set()
        for idx in exact_changed:
            lenient_changed_set.add(idx)
            if idx > 0:
                lenient_changed_set.add(idx - 1)
            if idx < total_time_points - 1:
                lenient_changed_set.add(idx + 1)

        # LIME explains one instance at a time. Grab the first row as a 1D array.
        instance_to_explain = X_test_2d[sample]

        explanation = explainer.explain_instance(
            data_row=instance_to_explain,
            predict_fn=rocket_predict,
            num_features=60
        )
        weights = np.zeros(60)

        for feature_idx, weight in explanation.local_exp[1]:
            weights[feature_idx] = np.abs(weight)

        # 3. Apply the matching 95th percentile cutoff threshold
        cutoff = np.percentile(weights, 95)
        shap_flagged = set(np.where(weights > cutoff)[0])

        # 4. Calculate confusion metrics
        s_tp = len(lenient_changed_set & shap_flagged)
        s_fp = len(shap_flagged - lenient_changed_set)
        s_fn = len(changed_set - shap_flagged)
            
        TP += s_tp

        # False Positives: Flagged points completely outside the lenient zone
        FP += s_fp

        # False Negatives: Actual changed points that were missed (even with the lenient zone)
        # To keep your metric behavior balanced, we look at which *exact* changes were missed completely
        FN += s_fn

    # Safe metric calculations
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    FDR = FP / (TP + FP) if (TP + FP) > 0 else 0.0
    precision_all.append(precision)
    recall_all.append(recall)
    f1_all.append(f1)
    fdr_all.append(FDR)

    

    print(
        f"Lenient Logic Result: "
        f"Precision={precision:.3f}, "
        f"Recall={recall:.3f}, "
        f"F1={f1:.3f}, "
        f"FDR={FDR:.3f}"
    )



In [ ]:
df = pd.DataFrame(
    {
        'recall': recall_all,
        'false discovery rate': fdr_all,
        'precision': precision_all,
        'f1': f1_all
    }
)
df.to_csv("C:/Users/satavans/Downloads/spike_files/lime_results/metrics_95p40-59.csv", index = False)

In [ ]:
for i in range(20,40):
    print(i)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from lime.lime_tabular import LimeTabularExplainer

precision_all = []
recall_all = []
fdr_all = []
f1_all = []
# Lime shapelet
for i in range(20,40):
    test_X = np.load("C:/Users/satavans/Downloads/shapelet_files/test_raw/X_" + str(i) + ".npy")
    test_X2 = np.load("C:/Users/satavans/Downloads/shapelet_files/test_raw/X2_" + str(i) + ".npy")
    train_X3 = np.load("C:/Users/satavans/Downloads/shapelet_files/train_raw/X3_" + str(i) + ".npy")
    y = pd.read_csv('C:/Users/satavans/Downloads/shapelet_files/train_files/shapelet_y' + str(i) + '.csv')['0']
    # fit the rocket transformer on sober drive data
    trf = Rocket(num_kernels=1000, random_state = 1234)
    classifier = LogisticRegression(max_iter = 2000)
    # 2. Fit pipeline (Assuming X_train is your time series data)
    X_train_transformed = trf.fit_transform(train_X3)
    scaler = StandardScaler()
    X_train_transformed = scaler.fit_transform(X_train_transformed)
    classifier.fit(X_train_transformed, y)

   # 3. Define the wrapper function 
    def rocket_predict(X_2d):
        # LIME passes (samples, time_steps). Inject the middle '1' dimension back for 3D ROCKET
        X_3d = X_2d.reshape(-1, 1, 60)
        X_tf = trf.transform(X_3d)
        X_tf = scaler.transform(X_tf)
        # LIME expects a 2D array of probabilities [prob_class_0, prob_class_1]
        # Use predict_proba instead of decision_function for classification
        return classifier.predict_proba(X_tf) 

    # 4. Prepare 2D background data and initialize LimeTabularExplainer
    # Squeeze 3D train data to 2D (samples, time_steps)
    X_train_2d = train_X3.squeeze(axis=1) 

    # Feature names represent each timestamp (from 0 to 59)
    feature_names = [f"time_{i}" for i in range(60)]

    explainer = LimeTabularExplainer(
        training_data=X_train_2d,       # Used to calculate mean/variance for feature perturbation
        mode="classification",          # Explaining a classifier
        feature_names=feature_names,                
        feature_selection = 'none'
    )

    # 5. Calculate LIME values for a specific instance
    # Squeeze 3D test data to 2D (samples, time_steps)
    X_test_2d = test_X2.squeeze(axis=1)

    

    TP = FP = FN = 0
    total_time_points = test_X.shape[2]
    for sample in range(test_X.shape[0]):
        # 1. Get the exact coordinates that changed
        exact_changed = np.where(np.abs(test_X[sample,0,:] - test_X2[sample,0,:]) > 0)[0]
        changed_set = set(exact_changed)

        # LIME explains one instance at a time. Grab the first row as a 1D array.
        instance_to_explain = X_test_2d[sample]

        explanation = explainer.explain_instance(
            data_row=instance_to_explain,
            predict_fn=rocket_predict,
            num_features=60
        )
        weights = np.zeros(60)

        for feature_idx, weight in explanation.local_exp[1]:
            weights[feature_idx] = np.abs(weight)

        # 3. Apply the matching 80th percentile cutoff threshold
        # cutoff = np.percentile(weights, 80)
        shap_flagged = set(np.where(weights >= (np.sort(weights)[::-1][9]))[0])
        # set(np.where(weights > cutoff)[0])

        # 4. Calculate confusion metrics
        s_tp = len(changed_set & shap_flagged)
        s_fp = len(shap_flagged - changed_set)
        s_fn = len(changed_set - shap_flagged)
            
        TP += s_tp

        # False Positives: Flagged points completely outside the lenient zone
        FP += s_fp

        # False Negatives: Actual changed points that were missed (even with the lenient zone)
        # To keep your metric behavior balanced, we look at which *exact* changes were missed completely
        FN += s_fn

    # Safe metric calculations
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    FDR = FP / (TP + FP) if (TP + FP) > 0 else 0.0
    precision_all.append(precision)
    recall_all.append(recall)
    f1_all.append(f1)
    fdr_all.append(FDR)

    

    print(
        f"Lenient Logic Result: "
        f"Precision={precision:.3f}, "
        f"Recall={recall:.3f}, "
        f"F1={f1:.3f}, "
        f"FDR={FDR:.3f}"
    )



In [ ]:
df = pd.DataFrame(
    {
        'recall': recall_all,
        'false discovery rate': fdr_all,
        'precision': precision_all,
        'f1': f1_all
    }
)
df.to_csv("C:/Users/satavans/Downloads/shapelet_files/lime_results/metrics_top10_20-39.csv", index = False)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import sys
import numpy as np
import pandas as pd
# LRP spike
sys.path.insert(0, r"C:\Users\satavans\Downloads\rocket-lrp")
from rocket_lrp.rocket import RocketTransform
from rocket_lrp.lrp import RocketLRPExplainer
from rocket_lrp.explainers import LinearSHAPExplainer


def filter_latent_importances(R_h, alpha_percent=30.0):
    """Zero the lowest alpha_percent of absolute latent importances per sample."""
    R_h = np.asarray(R_h, dtype=np.float64)
    if alpha_percent <= 0:
        return R_h
    if R_h.ndim == 1:
        R_h = R_h[None, :]

    out = R_h.copy()
    mag = np.abs(out)
    thr = np.percentile(mag, alpha_percent, axis=1, keepdims=True)
    out[mag < thr] = 0.0
    return out


def explain_with_paper_lrp(train_X3, test_X, test_X2, y, alpha_percent=30.0, epsilon=1e-6):
    """Run the paper's ROCKET + LinearSHAP + LRP workflow exactly for one split."""

    trf = Rocket(num_kernels=1000, random_state=1234)
    trf.fit(train_X3)

    rocket = RocketTransform.from_sktime(
        trf,
        train_X3
    )

    H_train = trf.transform(train_X3)
    scaler = StandardScaler().fit(H_train)
    H_train_scaled = scaler.transform(H_train)

    classifier = LogisticRegression(
        max_iter=1000,
    ).fit(H_train_scaled, y)

    # Stage 1: paper-style LinearSHAP in the latent ROCKET feature space.
    baseline = H_train_scaled.mean(axis=0)
    stage1 = LinearSHAPExplainer(
        classifier.coef_.ravel(),
        baseline=baseline,
        scaler_mean=scaler.mean_,
        scaler_scale=scaler.scale_,
    )

    H_test = trf.transform(test_X2)
    H_test_scaled = scaler.transform(H_test)
    R_h = stage1.explain(H_test_scaled)

    # Stage 2: paper-style LRP propagation to the original input space.
    lrp = RocketLRPExplainer(rocket, epsilon=epsilon, alpha=alpha_percent)
    R_h = filter_latent_importances(R_h, alpha_percent=alpha_percent)
    R_lrp = lrp.explain(test_X2, R_h)

    return classifier, scaler, trf, R_lrp


precision_all = []
recall_all = []
fdr_all = []
f1_all = []

for i in range(100):
    test_X = np.load("C:/Users/satavans/Downloads/spike_files_0.2spike/test_raw/X_" + str(i) + ".npy")
    test_X2 = np.load("C:/Users/satavans/Downloads/spike_files_0.2spike/test_raw/X2_" + str(i) + ".npy")
    train_X3 = np.load("C:/Users/satavans/Downloads/spike_files_0.2spike/train_raw/X3_" + str(i) + ".npy")
    y = pd.read_csv('C:/Users/satavans/Downloads/spike_files_0.2spike/train_files/spike_y' + str(i) + '.csv')['0']

    classifier, scaler, rocket, R_lrp = explain_with_paper_lrp(
        train_X3,
        test_X,
        test_X2,
        y,
        alpha_percent=30.0,
        epsilon=1e-6,
    )

    TP = FP = FN = 0
    total_time_points = test_X.shape[2]
    for sample in range(test_X.shape[0]):
        exact_changed = np.where(np.abs(test_X[sample, 0, :] - test_X2[sample, 0, :]) > 0)[0]
        changed_set = set(exact_changed)

        lenient_changed_set = set()
        for idx in exact_changed:
            lenient_changed_set.add(idx)
            if idx > 0:
                lenient_changed_set.add(idx - 1)
            if idx < total_time_points - 1:
                lenient_changed_set.add(idx + 1)

        rel = np.abs(R_lrp[sample, 0, :])
        top_k = len(changed_set)
        if len(changed_set) == 0:
            continue
        # max(1, int(np.ceil(0.05 * len(rel))))
        flagged = set(np.argsort(rel)[-top_k:])
        #s_tp = len(lenient_changed_set & flagged)
        #s_fp = len(flagged - lenient_changed_set)
        #s_fn = len(lenient_changed_set - flagged)

        matched_predictions = set()

        for true_idx in sorted(changed_set):
            matched = False
            
            for pred_idx in flagged:
                if pred_idx not in matched_predictions and abs(true_idx - pred_idx) <= 1:
                    TP += 1
                    matched_predictions.add(pred_idx)
                    matched = True
                    break

            if not matched:
                FN += 1

        for pred_idx in sorted(flagged):
            if pred_idx not in matched_predictions:
                FP += 1

    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    fdr = FP / (TP + FP) if (TP + FP) > 0 else 0.0

    precision_all.append(precision)
    recall_all.append(recall)
    fdr_all.append(fdr)
    f1_all.append(f1)

    print(
        f"Lenient Logic Result: "
        f"Precision={precision:.3f}, "
        f"Recall={recall:.3f}, "
        f"F1={f1:.3f}, "
        f"FDR={fdr:.3f}"
    )

print("Final mean precision:", np.mean(precision_all))
print("Final mean recall:", np.mean(recall_all))
print("Final mean FDR:", np.mean(fdr_all))
print("Final mean F1:", np.mean(f1_all))


In [ ]:
df = pd.DataFrame(
    {
        'recall': recall_all,
        'false discovery rate': fdr_all,
        'precision': precision_all,
        'f1': f1_all
    }
)
df.to_csv("C:/Users/satavans/Downloads/spike_files_0.2spike/lrp_results/metrics_topk.csv", index = False)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import sys
import numpy as np
import pandas as pd

sys.path.insert(0, r"C:\Users\satavans\Downloads\rocket-lrp")
from rocket_lrp.rocket import RocketTransform
from rocket_lrp.lrp import RocketLRPExplainer
from rocket_lrp.explainers import LinearSHAPExplainer


def filter_latent_importances(R_h, alpha_percent=30.0):
    """Zero the lowest alpha_percent of absolute latent importances per sample."""
    R_h = np.asarray(R_h, dtype=np.float64)
    if alpha_percent <= 0:
        return R_h
    if R_h.ndim == 1:
        R_h = R_h[None, :]

    out = R_h.copy()
    mag = np.abs(out)
    thr = np.percentile(mag, alpha_percent, axis=1, keepdims=True)
    out[mag < thr] = 0.0
    return out


def explain_with_paper_lrp(train_X3, test_X, test_X2, y, alpha_percent=30.0, epsilon=1e-6):
    """Run the paper's ROCKET + LinearSHAP + LRP workflow exactly for one split."""

    trf = Rocket(num_kernels=1000, random_state=1234)
    trf.fit(train_X3)

    rocket = RocketTransform.from_sktime(
        trf,
        train_X3
    )

    H_train = trf.transform(train_X3)
    scaler = StandardScaler().fit(H_train)
    H_train_scaled = scaler.transform(H_train)

    classifier = LogisticRegression(
        max_iter=1000,
    ).fit(H_train_scaled, y)

    # Stage 1: paper-style LinearSHAP in the latent ROCKET feature space.
    baseline = H_train_scaled.mean(axis=0)
    stage1 = LinearSHAPExplainer(
        classifier.coef_.ravel(),
        baseline=baseline,
        scaler_mean=scaler.mean_,
        scaler_scale=scaler.scale_,
    )

    H_test = trf.transform(test_X2)
    H_test_scaled = scaler.transform(H_test)
    R_h = stage1.explain(H_test_scaled)

    # Stage 2: paper-style LRP propagation to the original input space.
    lrp = RocketLRPExplainer(rocket, epsilon=epsilon, alpha=alpha_percent)
    R_h = filter_latent_importances(R_h, alpha_percent=alpha_percent)
    R_lrp = lrp.explain(test_X2, R_h)

    return classifier, scaler, trf, R_lrp


precision_all = []
recall_all = []
fdr_all = []
f1_all = []
# LRP shapelet
for i in range(100):
    test_X = np.load("C:/Users/satavans/Downloads/shapelet_files/test_raw/X_" + str(i) + ".npy")
    test_X2 = np.load("C:/Users/satavans/Downloads/shapelet_files/test_raw/X2_" + str(i) + ".npy")
    train_X3 = np.load("C:/Users/satavans/Downloads/shapelet_files/train_raw/X3_" + str(i) + ".npy")
    y = pd.read_csv('C:/Users/satavans/Downloads/shapelet_files/train_files/shapelet_y' + str(i) + '.csv')['0']

    classifier, scaler, rocket, R_lrp = explain_with_paper_lrp(
        train_X3,
        test_X,
        test_X2,
        y,
        alpha_percent=30.0,
        epsilon=1e-6,
    )

    TP = FP = FN = 0
    total_time_points = test_X.shape[2]
    for sample in range(test_X.shape[0]):
        exact_changed = np.where(np.abs(test_X[sample, 0, :] - test_X2[sample, 0, :]) > 0)[0]
        changed_set = set(exact_changed)


        rel = np.abs(R_lrp[sample, 0, :])
        top_k = 10
        flagged = set(np.argsort(rel)[-top_k:])
        s_tp = len(changed_set & flagged)
        s_fp = len(flagged - changed_set)
        s_fn = len(changed_set - flagged)

        TP += s_tp
        FP += s_fp
        FN += s_fn
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    fdr = FP / (TP + FP) if (TP + FP) > 0 else 0.0

    precision_all.append(precision)
    recall_all.append(recall)
    fdr_all.append(fdr)
    f1_all.append(f1)

    print(
        f"Lenient Logic Result: "
        f"Precision={precision:.3f}, "
        f"Recall={recall:.3f}, "
        f"F1={f1:.3f}, "
        f"FDR={fdr:.3f}"
    )

print("Final mean precision:", np.mean(precision_all))
print("Final mean recall:", np.mean(recall_all))
print("Final mean FDR:", np.mean(fdr_all))
print("Final mean F1:", np.mean(f1_all))


In [ ]:
df2 = pd.read_csv("C:/Users/satavans/Downloads/spike_files_0.2spike/shap_results/metrics_topk.csv")
print("Mean F1 = ", np.mean(df2['recall']))
print("STD F1 = ", np.std(df2['recall']))

In [ ]:
df1 = pd.read_csv("C:/Users/satavans/Downloads/spike_files_0.2spike/results/metrics_topk_0.0001.csv")
print("Mean F1 = ", np.mean(df1['f1']))
print("STD F1 = ", np.std(df1['f1']))

In [ ]:
from scipy.stats import ttest_rel

# Example: F1 scores from 100 identical trials
# f1_sum = [your 100 F1 scores for your method]
# f1_lrp = [your 100 F1 scores for ROCKET-LRP]

t_stat, p_value = ttest_rel(df1['recall'], df2['recall'])

print("t-statistic:", t_stat)
print("p-value:", p_value)


In [ ]:
np.percentile(df1['f1'], [90,50,10,0])

In [ ]:
np.percentile(df2['f1'], [90,50,10,0])

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

# --- 1. LOAD YOUR DATA ---
# Replace this section with your actual data arrays or DataFrame columns
# df = pd.read_csv("explainability_results.csv")
method_A = df1['recall'].values
method_B = df2['recall'].values

# --- 2. RUN THE COMPARATIVE STATS ---
# Paired T-Test (Vulnerable to the 7 bad values)
t_stat, t_pvalue = stats.ttest_rel(method_A, method_B)

# Wilcoxon Signed-Rank Test (Robust to the 7 bad values)
w_stat, w_pvalue = stats.wilcoxon(method_A, method_B, alternative='less') 
# Note: alternative='less' tests if Method A scores are significantly LESS than Method B scores.

# --- 3. DISPLAY RESULTS ---
print(f"Method A Mean Score: {np.mean(method_A):.4f}")
print(f"Method B Mean Score: {np.mean(method_B):.4f}")
print("-" * 50)
print(f"Paired T-Test p-value: {t_pvalue:.4f}")
print(f"Wilcoxon Test p-value: {w_pvalue:.4f}")
